In [1]:
# bowaka_v2_lab notebook bootstrap cell — DO NOT EDIT BY HAND.
# Adds the lab's src/ (and its bowaka_common dependency) to sys.path and pins
# the working directory to the repo root, so `import bowaka_v2_lab` and
# repo-root-relative CONFIG_PATH parameters resolve identically under jupyter,
# papermill, and the QuantsLab scheduler.
import os
import sys
from pathlib import Path

_lab_root = None
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "bowaka_v2_lab" / "__init__.py").is_file():
        _lab_root = _candidate
        break
if _lab_root is None:
    raise RuntimeError(
        f"bowaka_v2_lab bootstrap: src/bowaka_v2_lab/ not found at or above {Path.cwd()}"
    )

# Pin CWD to the repo root (the directory holding research_notebooks/ and the
# Makefile) so repo-root-relative CONFIG_PATH values resolve regardless of how
# the notebook was launched (jupyter CWD = notebook dir, scheduler = repo root).
_repo_root = _lab_root
for _candidate in [_lab_root, *_lab_root.parents]:
    if (_candidate / "research_notebooks").is_dir() and (_candidate / "Makefile").is_file():
        _repo_root = _candidate
        break
os.chdir(_repo_root)

# Make the lab and its bowaka_common dependency importable from the working
# tree, even when the packages are not pip-installed. v1 bowaka_lab is
# deliberately excluded — v2 must not import v1.
for _src in (_lab_root / "src",
             _repo_root / "research_notebooks" / "bowaka_common" / "src"):
    if _src.is_dir() and str(_src) not in sys.path:
        sys.path.insert(0, str(_src))

import bowaka_v2_lab  # noqa: F401
print(f"bowaka_v2_lab {bowaka_v2_lab.__version__} (cwd={_repo_root})")


bowaka_v2_lab 0.1.0 (cwd=/quants-lab)


In [2]:
# Papermill parameter cell.
CONFIG_PATH = 'research_notebooks/bowaka_v2_lab/configs/bowaka_v2_backtest_smoke.yml'


# 08 — Counterfactual Exits & Holds

Keeps entries fixed and re-runs real backtests with alternative exit / hold parameters to compare outcomes.

In [3]:
from bowaka_v2_lab.config import load_config
from bowaka_v2_lab.backtest_runner import run_config_backtest
from bowaka_v2_lab.research.counterfactuals import run_counterfactual_grid
base_cfg = load_config(CONFIG_PATH)
result = run_counterfactual_grid(
    base_cfg=base_cfg,
    exit_variants=[{'max_hold_days': 1}, {'max_hold_days': 5}, {'take_profit_pct': 0.10}],
    backtest_runner=lambda cfg: run_config_backtest(cfg).summary,
)
print(result)


   variant_idx  max_hold_days  schema_version  \
0            0            1.0               1   
1            1            5.0               1   
2            2            NaN               1   

                                          run_id strategy_id strategy_version  \
0  20260524_bowaka_v2_backtest_cea851ce_abf783c9   bowaka_v2            0.1.0   
1  20260524_bowaka_v2_backtest_ffddaa73_86cce705   bowaka_v2            0.1.0   
2  20260524_bowaka_v2_backtest_8ada5e9f_b6603853   bowaka_v2            0.1.0   

  feed cost_stress  initial_bankroll  final_bankroll  ...  partial_fill_count  \
0  iex        base          100000.0    99889.457356  ...                   0   
1  iex        base          100000.0    99889.457356  ...                   0   
2  iex        base          100000.0    99889.457356  ...                   0   

   fill_rate  historical_quote_coverage_pct  fees_paid_total  \
0        1.0                            0.0              0.0   
1        1.0             